In [ ]:
# 1. Setup paths and mount Drive
from google.colab import drive
from pathlib import Path
import os

# Mount Drive
drive.mount('/content/drive')

# Path definitions - Drive (persistent)
WORKSPACE = '/content/drive/MyDrive/evo1_ablation'
RESULTS_DIR = Path(f'{WORKSPACE}/Results')

# Path definitions - Local (ephemeral)
CHECKPOINTS_DIR = Path('/content/checkpoints')
LOGS_DIR = Path('/content/logs')

# Create directories
for d in [RESULTS_DIR, CHECKPOINTS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Clear PYTHONPATH
os.environ.pop('PYTHONPATH', None)
os.environ['MUJOCO_GL'] = 'egl'
os.environ.pop('DISPLAY', None)

# Check GPU
import torch
if not torch.cuda.is_available():
    raise RuntimeError("❌ No GPU! Enable in Runtime > Change runtime type")

gpu_name = torch.cuda.get_device_name(0)
total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"✅ GPU: {gpu_name} ({total_mem:.1f} GB)")
print(f"✅ Workspace: {WORKSPACE}")
print(f"✅ Results: {RESULTS_DIR}")

Mounted at /content/drive
✅ GPU: NVIDIA L4 (22.0 GB)
✅ Workspace: /content/drive/MyDrive/evo1_ablation
✅ Results: /content/drive/MyDrive/evo1_ablation/Results


In [ ]:
# 2. Install dependencies (3 SEPARATE conda environments - official structure)
print('📦 Installing system dependencies...')
!apt-get update -qq
!apt-get install -y -qq git wget ffmpeg libosmesa6-dev patchelf libgl1-mesa-glx libegl1-mesa > /dev/null 2>&1

print('📦 Installing Miniconda...')
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
!bash /tmp/miniconda.sh -b -p /opt/conda > /dev/null 2>&1
!rm /tmp/miniconda.sh
os.environ['PATH'] = f"/opt/conda/bin:{os.environ['PATH']}"
!conda init bash
!conda config --set always_yes yes

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

print('\n📦 Creating 3 conda environments (official Evo-1 structure)...')
print('='*60)

# Environment 1: Evo-1 server (Python 3.10)
print('\n[1/3] evo1_server (Python 3.10) - for Evo-1 model')
!conda create -n evo1_server python=3.10 -y
!conda run -n evo1_server pip install  \
  torch==2.5.1+cu121 torchvision==0.20.1+cu121 \
  --index-url https://download.pytorch.org/whl/cu121

# LOG: 28 Jan 2026 transformers == 5.0.0 (latest currently) causes issue with server initialisation
# specifically, issue with meta tensors and internvit initialisation.
# Use previous version 4.57.6 for functionability

!conda run -n evo1_server pip install \
  'numpy>=1.26.4,<2.0' 'transformers==4.57.6' accelerate diffusers \
  einops timm pillow opencv-python-headless \
  websockets pyyaml huggingface_hub tqdm
print('📦 Installing flash-attn (required, ~10-15 min)...')
!conda run -n evo1_server pip install flash-attn --no-build-isolation 2>&1 | tail -5 || echo '⚠️ Flash-attn failed'
print('✅ evo1_server ready')

# Environment 2: LIBERO client (Python 3.8.13 - OFFICIAL requirement)
print('\n[2/3] libero_client (Python 3.8.13) - for LIBERO benchmark')
!conda create -n libero_client python=3.8.13 -y
!conda run -n libero_client pip install \
  'numpy>=1.20,<2.0' robosuite==1.4.1 mujoco==2.3.7 \
  imageio h5py bddl websockets huggingface_hub
!conda run -n libero_client pip install \
  torch==1.11.0+cu113 torchvision==0.12.0+cu113 torchaudio==0.11.0 \
  --extra-index-url https://download.pytorch.org/whl/cu113
print('✅ libero_client ready')

# Environment 3: MetaWorld client (Python 3.10)
print('\n[3/3] metaworld_client (Python 3.10) - for MetaWorld benchmark')
!conda create -n metaworld_client python=3.10 -y
!conda run -n metaworld_client pip install \
  mujoco websockets opencv-python packaging huggingface_hub metaworld gymnasium

print('✅ metaworld_client ready')

print('\n' + '='*60)
print('✅ All 3 environments created!')
!conda run -n evo1_server python -c "import sys; print(f'  evo1_server: Python {sys.version.split()[0]}')"
!conda run -n libero_client python -c "import sys; print(f'  libero_client: Python {sys.version.split()[0]}')"
!conda run -n metaworld_client python -c "import sys; print(f'  metaworld_client: Python {sys.version.split()[0]}')"


📦 Installing system dependencies...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
📦 Installing Miniconda...
no change     /opt/conda/condabin/conda
no change     /opt/conda/bin/conda
no change     /opt/conda/bin/conda-env
no change     /opt/conda/bin/activate
no change     /opt/conda/bin/deactivate
no change     /opt/conda/etc/profile.d/conda.sh
no change     /opt/conda/etc/fish/conf.d/conda.fish
no change     /opt/conda/shell/condabin/Conda.psm1
no change     /opt/conda/shell/condabin/conda-hook.ps1
no change     /opt/conda/lib/python3.13/site-packages/xontrib/conda.xsh
no change     /opt/conda/etc/profile.d/conda.csh
modified      /root/.bashrc

==> For changes to take effect, close and re-open your current shell. <==

accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs

In [ ]:
# 3. Clone and install repositories in their respective environments
import os
import yaml
from pathlib import Path

print('📦 Setting up repositories...')
print('='*60)

# ==================== Clone Evo-1 ====================
print('\n[1/3] Cloning Evo-1...')
if not Path('/content/Evo-1/.git').exists():
    !git clone https://github.com/MINT-SJTU/Evo-1.git /content/Evo-1
    print('✅ Cloned Evo-1')
else:
    print('✅ Evo-1 already cloned')

# Install Evo-1 dependencies in server environment
print('\n📦 Installing Evo-1 dependencies in evo1_server...')
evo1_requirements = Path('/content/Evo-1/Evo_1/requirements.txt')
if evo1_requirements.exists():
    !conda run -n evo1_server pip install -q -r /content/Evo-1/Evo_1/requirements.txt
    print('✅ Evo-1 dependencies installed in evo1_server')
else:
    print('⚠️ WARNING: Evo-1 requirements.txt not found')

# ==================== Clone LIBERO ====================
print('\n[2/3] Cloning LIBERO...')
if not Path('/content/LIBERO_evaluation/LIBERO/.git').exists():
    !git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git /content/LIBERO_evaluation/LIBERO
    print('✅ Cloned LIBERO')
else:
    print('✅ LIBERO already cloned')

# Install LIBERO in client environment (Python 3.8.13)
print('\n📦 Installing LIBERO in libero_client...')
libero_requirements = Path('/content/LIBERO_evaluation/LIBERO/requirements.txt')
if libero_requirements.exists():
    !conda run -n libero_client pip install -q -r /content/LIBERO_evaluation/LIBERO/requirements.txt
    !conda run -n libero_client pip install -q -e /content/LIBERO_evaluation/LIBERO
    print('✅ LIBERO installed in libero_client')
else:
    print('⚠️ WARNING: LIBERO requirements.txt not found')

# ==================== Configure LIBERO ====================
print('\n[3/3] Configuring LIBERO...')
os.makedirs(os.path.expanduser('~/.libero'), exist_ok=True)
libero_cfg = os.path.expanduser('~/.libero/config.yaml')

if not os.path.exists(libero_cfg):
    cfg = {
        'benchmark_root': '/content/LIBERO_evaluation/LIBERO/libero/libero',
        'bddl_files': '/content/LIBERO_evaluation/LIBERO/libero/libero/bddl_files',
        'init_states': '/content/LIBERO_evaluation/LIBERO/libero/libero/init_files',
        'datasets': '/content/LIBERO_evaluation/LIBERO/datasets',
        'assets': '/content/LIBERO_evaluation/LIBERO/libero/libero/assets'
    }
    with open(libero_cfg, 'w') as f:
        yaml.dump(cfg, f)
    print('✅ LIBERO config created at ~/.libero/config.yaml')
else:
    print('✅ LIBERO config already exists')

# ==================== Install MetaWorld ====================
print('\n📦 Installing MetaWorld in metaworld_client...')
!conda run -n metaworld_client pip install -q metaworld
!conda run -n metaworld_client pip install -q opencv-python
print('✅ MetaWorld and dependencies installed')

# ==================== Verification ====================
print('\n' + '='*60)
print('🔍 Verifying installations...')
print('='*60)

verification_commands = [
    ('evo1_server', 'python -c "import torch; print(f\'PyTorch: {torch.__version__}\')"'),
    ('libero_client', 'python -c "import libero; print(\'LIBERO imported successfully\')"'),
    ('metaworld_client', 'python -c "import metaworld; print(\'MetaWorld imported successfully\')"'),
]

for env_name, cmd in verification_commands:
    print(f'\n{env_name}:')
    !conda run -n {env_name} {cmd}

print('\n' + '='*60)
print('✅ All repositories installed and configured!')
print('='*60)

📦 Setting up repositories...

[1/3] Cloning Evo-1...
Cloning into '/content/Evo-1'...
remote: Enumerating objects: 731, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 731 (delta 56), reused 50 (delta 50), pack-reused 663 (from 1)
Receiving objects: 100% (731/731), 6.47 MiB | 14.29 MiB/s, done.
Resolving deltas: 100% (198/198), done.
✅ Cloned Evo-1

📦 Installing Evo-1 dependencies in evo1_server...
✅ Evo-1 dependencies installed in evo1_server

[2/3] Cloning LIBERO...
Cloning into '/content/LIBERO_evaluation/LIBERO'...
remote: Enumerating objects: 1788, done.
remote: Counting objects: 100% (582/582), done.
remote: Compressing objects: 100% (210/210), done.
remote: Total 1788 (delta 380), reused 372 (delta 372), pack-reused 1206 (from 1)
Receiving objects: 100% (1788/1788), 315.98 MiB | 34.17 MiB/s, done.
Resolving deltas: 100% (755/755), done.
Updating files: 100% (1116/1116), done.
✅ Cloned LIBERO

📦 Installing LIBERO 

In [ ]:
# 4. Download checkpoints
from huggingface_hub import snapshot_download

CHECKPOINTS_DIR = Path('/content/checkpoints/')

CHECKPOINTS = {
    'libero': {'repo': 'MINT-SJTU/Evo1_LIBERO', 'dir': CHECKPOINTS_DIR / 'libero'},
    'metaworld': {'repo': 'MINT-SJTU/Evo1_MetaWorld', 'dir': CHECKPOINTS_DIR / 'metaworld'}
}

print('📥 Downloading checkpoints...')
for name, cfg in CHECKPOINTS.items():
    cfg['dir'].mkdir(parents=True, exist_ok=True)
    model_file = cfg['dir'] / 'mp_rank_00_model_states.pt'

    if model_file.exists() and model_file.stat().st_size > 1_000_000:
        print(f"✅ {name}: {model_file.stat().st_size / 1e9:.2f} GB")
    else:
        print(f"⏳ Downloading {name}...")
        snapshot_download(
            repo_id=cfg['repo'],
            local_dir=str(cfg['dir']),
            local_dir_use_symlinks=False,
        )
        print(f"✅ {name} downloaded")

print('\n✅ Checkpoints ready')

📥 Downloading checkpoints...
⏳ Downloading libero...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

✅ libero downloaded
⏳ Downloading metaworld...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

✅ metaworld downloaded

✅ Checkpoints ready


In [ ]:
# Create ablated server - ONLY difference: passes zeros to state encoder
ablated_server = '''#!/usr/bin/env python3
"""Evo-1 Server with STATE ABLATION - Zero State Input to Encoder"""
import sys
import os
import asyncio
import websockets
import numpy as np
import cv2
import json
import torch
from PIL import Image
from torchvision import transforms
import argparse

# Command-line arguments
parser = argparse.ArgumentParser()
parser.add_argument("--port", type=int, required=True, help="WebSocket server port")
parser.add_argument("--checkpoint", type=str, required=True, help="Path to model checkpoint directory")
parser.add_argument("--name", type=str, default="evo1_ablation", help="Server name for logging")
args = parser.parse_args()

print(f"[{args.name}] STATE ABLATION: Zero state input to encoder")
print(f"[{args.name}] Port: {args.port}")
print(f"[{args.name}] Checkpoint: {args.checkpoint}")

sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
from scripts.Evo1 import EVO1


class Normalizer:
    def __init__(self, stats_or_path):
        if isinstance(stats_or_path, str):
            with open(stats_or_path, "r") as f:
                stats = json.load(f)
        else:
            stats = stats_or_path

        def pad_to_24(x):
            x = torch.tensor(x, dtype=torch.float32)
            if x.shape[0] < 24:
                pad = torch.zeros(24 - x.shape[0], dtype=torch.float32)
                x = torch.cat([x, pad], dim=0)
            elif x.shape[0] > 24:
                raise ValueError(f"Input length {x.shape[0]} exceeds expected 24")
            return x

        if len(stats) != 1:
            raise ValueError(f"norm_stats.json should contain only one robot key, but: {list(stats.keys())}")

        robot_key = list(stats.keys())[0]
        robot_stats = stats[robot_key]

        self.state_min = pad_to_24(robot_stats["observation.state"]["min"])
        self.state_max = pad_to_24(robot_stats["observation.state"]["max"])
        self.action_min = pad_to_24(robot_stats["action"]["min"])
        self.action_max = pad_to_24(robot_stats["action"]["max"])

    def normalize_state(self, state: torch.Tensor) -> torch.Tensor:
        """ABLATION: Return zeros instead of normalized state"""
        # Keep same shape/device/dtype, but all zeros
        return torch.zeros_like(state)

    def denormalize_action(self, action: torch.Tensor) -> torch.Tensor:
        action_min = self.action_min.to(action.device, dtype=action.dtype)
        action_max = self.action_max.to(action.device, dtype=action.dtype)
        if action.ndim == 1:
            action = action.view(1, -1)
        return (action + 1.0) / 2.0 * (action_max - action_min + 1e-8) + action_min


def load_model_and_normalizer(ckpt_dir):

    config = json.load(open(os.path.join(ckpt_dir, "config.json")))
    stats = json.load(open(os.path.join(ckpt_dir, "norm_stats.json")))

    config["finetune_vlm"] = False
    config["finetune_action_head"] = False
    config["num_inference_timesteps"] = 32

    model = EVO1(config).eval()
    ckpt_path = os.path.join(ckpt_dir, "mp_rank_00_model_states.pt")

    checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    model.load_state_dict(checkpoint["module"], strict=True)
    model = model.to("cuda")

    normalizer = Normalizer(stats)
    return model, normalizer


def decode_image_from_list(img_list):
    img_array = np.array(img_list, dtype=np.uint8)
    img = cv2.resize(img_array, (448, 448))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(img)
    return transforms.ToTensor()(pil).to("cuda")


def infer_from_json_dict(data: dict, model, normalizer):
    device = "cuda"
    model_dtype = next(model.parameters()).dtype

    # Process images (same as original)
    images = [decode_image_from_list(img) for img in data["image"]]
    assert len(images) == 3, "Must provide exactly 3 images."
    for img in images:
        assert img.shape == (3, 448, 448), "image_size must be (3,448,448)"

    # Process state (same as original, but will be zeroed in normalize_state)
    state = torch.tensor(data["state"], dtype=torch.float32, device=device)
    if state.ndim == 1:
        state = state.unsqueeze(0)
    if state.shape[1] < 24:
        state = torch.cat([state, torch.zeros((1, 24 - state.shape[1]), device=device)], dim=1)

    # ABLATION: This returns zeros instead of normalized state
    norm_state = normalizer.normalize_state(state).to(dtype=torch.float32)

    # Masks and prompt (same as original)
    prompt = data["prompt"]
    image_mask = torch.tensor(data["image_mask"], dtype=torch.int32, device=device)
    action_mask = torch.tensor([data["action_mask"]], dtype=torch.int32, device=device)

    # Inference (same as original)
    with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
        action = model.run_inference(
            images=images,
            image_mask=image_mask,
            prompt=prompt,
            state_input=norm_state,  # ⚠️ This is all zeros due to ablation
            action_mask=action_mask
        )
        action = action.reshape(1, -1, 24)
        action = normalizer.denormalize_action(action[0])
        return action.cpu().numpy().tolist()


async def handle_request(websocket, model, normalizer):
    print("Client connected")
    try:
        async for message in websocket:
            json_data = json.loads(message)
            print(f"Received JSON observation (STATE ABLATION ACTIVE)")
            actions = infer_from_json_dict(json_data, model, normalizer)
            await websocket.send(json.dumps(actions))
            print("Sent action chunk")

    except websockets.exceptions.ConnectionClosed:
        print("Client disconnected.")
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()


if __name__ == "__main__":
    print(f"\\nLoading EVO_1 model with STATE ABLATION...")
    model, normalizer = load_model_and_normalizer(args.checkpoint)
    print(f"✅ Model loaded - state encoder will receive ZEROS only\\n")

    async def main():
        print(f"🚀 Ablated server running at ws://0.0.0.0:{args.port}")
        print("=" * 60)
        async with websockets.serve(
            lambda ws: handle_request(ws, model, normalizer),
            "0.0.0.0", args.port,
            max_size=100_000_000,
            ping_interval=120,      # Send ping every 120 seconds
            ping_timeout=300        # Wait 300 seconds for pong response
        ):
            await asyncio.Future()

    asyncio.run(main())
'''

with open('/content/Evo-1/Evo_1/scripts/Evo1_ablated_server.py', 'w') as f:
    f.write(ablated_server)

print('✅ Ablated server script created at: /content/Evo-1/Evo_1/scripts/Evo1_ablated_server.py')
print('   ONLY difference from baseline: normalize_state() returns zeros')
print('   Everything else identical: same inputs, same inference flow, same outputs')
print('   Tests: Impact of removing proprioceptive state information')
print('\n📋 Usage:')
print('   python Evo1_ablated_server.py --port 9001 --checkpoint /path/to/checkpoint --name ablation_trial_1')
print('\n🔧 Arguments:')
print('   --port: WebSocket server port (required)')
print('   --checkpoint: Path to model checkpoint directory (required)')
print('   --name: Server name for logging (optional, default: evo1_ablation)')

✅ Ablated server script created at: /content/Evo-1/Evo_1/scripts/Evo1_ablated_server.py
   ONLY difference from baseline: normalize_state() returns zeros
   Everything else identical: same inputs, same inference flow, same outputs
   Tests: Impact of removing proprioceptive state information

📋 Usage:
   python Evo1_ablated_server.py --port 9001 --checkpoint /path/to/checkpoint --name ablation_trial_1

🔧 Arguments:
   --port: WebSocket server port (required)
   --checkpoint: Path to model checkpoint directory (required)
   --name: Server name for logging (optional, default: evo1_ablation)


In [ ]:
# 6. Create and patch client scripts for 5 parallel trials
import re
import shutil
from pathlib import Path

# Configuration
NUM_TRIALS = 10
LIBERO_BASE_PORT = 9001  # Ports: 9001-9005
METAWORLD_BASE_PORT = 9101  # Ports: 9101-9105

print('📝 Creating client scripts for parallel trials...')
print(f'   Creating {NUM_TRIALS} LIBERO clients and {NUM_TRIALS} MetaWorld clients')
print('='*60)

# ==================== Patch LIBERO Clients ====================
print(f'\n[1/2] Patching {NUM_TRIALS} LIBERO client scripts...')

libero_client_base = '/content/Evo-1/LIBERO_evaluation/libero_client_4tasks.py'

if not Path(libero_client_base).exists():
    print(f'⚠️ WARNING: {libero_client_base} not found! Skipping LIBERO patching.')
else:
    with open(libero_client_base, 'r') as f:
        content = f.read()

    # Apply base patches
    content = content.replace('os.environ["MUJOCO_GL"] = "osmesa"', 'os.environ["MUJOCO_GL"] = "egl"')

    if 'max_size=' not in content:
        content = re.sub(
            r'async with websockets\.connect\(([^,\)]+)\)',
            r'''async with websockets.connect(
                  SERVER_URL,
                  max_size=100*1024*1024,
                  ping_interval=120,
                  ping_timeout=300
              )''',
            content
        )

    # Save base version
    with open(libero_client_base, 'w') as f:
        f.write(content)

    # Create trial-specific versions
    for i in range(1, NUM_TRIALS + 1):
        libero_client_patched = f'/content/Evo-1/LIBERO_evaluation/libero_client_trial_{i}.py'
        shutil.copy(libero_client_base, libero_client_patched)

        port = LIBERO_BASE_PORT + i - 1

        with open(libero_client_patched, 'r') as f:
            content = f.read()

        # Patch server URL for this trial
        content = re.sub(
            r'SERVER_URL\s*=\s*"[^"]*"',
            f'SERVER_URL = "ws://localhost:{port}"',
            content
        )

        content = re.sub(
            r'async with websockets\.connect\(([^,\)]+)\)',
            r'async with websockets.connect(\1, max_size=100*1024*1024, ping_interval=120, ping_timeout=300)',
            content
        )

        content = re.sub(
            r'ckpt_name\s*=\s*f?"Evo1_libero_all"',
            f'ckpt_name = "Evo1_libero_trial_{i}"',
            content
        )

        with open(libero_client_patched, 'w') as f:
            f.write(content)

        print(f'  ✅ libero_client_trial_{i}.py → port {port}')

# ==================== Patch MetaWorld Clients ====================
print(f'\n[2/2] Patching {NUM_TRIALS} MetaWorld client scripts...')

mw_client_base = '/content/Evo-1/MetaWorld_evaluation/mt50_evo1_client_prompt.py'

if not Path(mw_client_base).exists():
    print(f'⚠️ WARNING: {mw_client_base} not found! Skipping MetaWorld patching.')
else:
    # First, patch the base MetaWorld client
    with open(mw_client_base, 'r') as f:
        content = f.read()

    # Apply base patches
    content = re.sub(
        r'SHOW_WINDOW\s*=\s*True',
        'SHOW_WINDOW = False',
        content
    )

    # Fix ORDER_JSON_PATH to absolute path
    content = re.sub(
        r'ORDER_JSON_PATH\s*=\s*"mt50_order\.json"',
        'ORDER_JSON_PATH = "/content/Evo-1/MetaWorld_evaluation/mt50_order.json"',
        content
    )

    # Fix TASKS_JSONL_PATH to absolute path
    content = re.sub(
        r'TASKS_JSONL_PATH\s*=\s*"tasks\.jsonl"',
        'TASKS_JSONL_PATH = "/content/Evo-1/MetaWorld_evaluation/tasks.jsonl"',
        content
    )

    # Fix websocket max_size
    if 'max_size=' not in content:
        content = re.sub(
            r'async with websockets\.connect\(server_url\)',
            r'async with websockets.connect(server_url, max_size=100_000_000)',
            content
        )

    # Save base version
    with open(mw_client_base, 'w') as f:
        f.write(content)

    # Create trial-specific versions
    for i in range(1, NUM_TRIALS + 1):
        mw_client_patched = f'/content/Evo-1/MetaWorld_evaluation/mt50_evo1_client_trial_{i}.py'
        shutil.copy(mw_client_base, mw_client_patched)

        port = METAWORLD_BASE_PORT + i - 1

        with open(mw_client_patched, 'r') as f:
            content = f.read()

        # Patch server URL for this trial
        content = re.sub(
            r'SERVER_URL\s*=\s*"[^"]*"',
            f'SERVER_URL = "ws://localhost:{port}"',
            content
        )

        # Update log paths to be trial-specific
        content = re.sub(
            r'LOG_DIR\s*=\s*"[^"]*"',
            f'LOG_DIR = "/content/metaworld_logs_trial_{i}"',
            content
        )

        content = re.sub(
            r'VIDEO_SAVE_DIR\s*=\s*"[^"]*"',
            f'VIDEO_SAVE_DIR = "/content/metaworld_videos_trial_{i}"',
            content
        )

        content = re.sub(
            r'INSPECT_DIR\s*=\s*"[^"]*"',
            f'INSPECT_DIR = "/content/metaworld_frames_trial_{i}"',
            content
        )

        with open(mw_client_patched, 'w') as f:
            f.write(content)

        print(f'  ✅ mt50_evo1_client_trial_{i}.py → port {port}')

print('\n' + '='*60)
print(f'✅ All client scripts ready!')
print(f'   LIBERO: {NUM_TRIALS} clients (ports {LIBERO_BASE_PORT}-{LIBERO_BASE_PORT + NUM_TRIALS - 1})')
print(f'   MetaWorld: {NUM_TRIALS} clients (ports {METAWORLD_BASE_PORT}-{METAWORLD_BASE_PORT + NUM_TRIALS - 1})')
print('='*60)

# ==================== Verify Patches ====================
print('\n🔍 Verifying patches...')

# Verify LIBERO
for i in range(1, NUM_TRIALS + 1):
    client_path = Path(f'/content/Evo-1/LIBERO_evaluation/libero_client_trial_{i}.py')
    if client_path.exists():
        with open(client_path, 'r') as f:
            content = f.read()
        expected_port = LIBERO_BASE_PORT + i - 1
        if f'ws://localhost:{expected_port}' in content:
            print(f'  ✅ LIBERO trial {i}: Correct port {expected_port}')
        else:
            print(f'  ❌ LIBERO trial {i}: Port patch failed!')
    else:
        print(f'  ❌ LIBERO trial {i}: File not created!')

# Verify MetaWorld
for i in range(1, NUM_TRIALS + 1):
    client_path = Path(f'/content/Evo-1/MetaWorld_evaluation/mt50_evo1_client_trial_{i}.py')
    if client_path.exists():
        with open(client_path, 'r') as f:
            content = f.read()
        expected_port = METAWORLD_BASE_PORT + i - 1
        if f'ws://localhost:{expected_port}' in content:
            print(f'  ✅ MetaWorld trial {i}: Correct port {expected_port}')
        else:
            print(f'  ❌ MetaWorld trial {i}: Port patch failed!')
    else:
        print(f'  ❌ MetaWorld trial {i}: File not created!')

print('\n✅ Patching verification complete!')

📝 Creating client scripts for parallel trials...
   Creating 10 LIBERO clients and 10 MetaWorld clients

[1/2] Patching 10 LIBERO client scripts...
  ✅ libero_client_trial_1.py → port 9001
  ✅ libero_client_trial_2.py → port 9002
  ✅ libero_client_trial_3.py → port 9003
  ✅ libero_client_trial_4.py → port 9004
  ✅ libero_client_trial_5.py → port 9005
  ✅ libero_client_trial_6.py → port 9006
  ✅ libero_client_trial_7.py → port 9007
  ✅ libero_client_trial_8.py → port 9008
  ✅ libero_client_trial_9.py → port 9009
  ✅ libero_client_trial_10.py → port 9010

[2/2] Patching 10 MetaWorld client scripts...
  ✅ mt50_evo1_client_trial_1.py → port 9101
  ✅ mt50_evo1_client_trial_2.py → port 9102
  ✅ mt50_evo1_client_trial_3.py → port 9103
  ✅ mt50_evo1_client_trial_4.py → port 9104
  ✅ mt50_evo1_client_trial_5.py → port 9105
  ✅ mt50_evo1_client_trial_6.py → port 9106
  ✅ mt50_evo1_client_trial_7.py → port 9107
  ✅ mt50_evo1_client_trial_8.py → port 9108
  ✅ mt50_evo1_client_trial_9.py → port 910

In [ ]:
# 7. Run benchmarks - Sequential execution (MetaWorld first, then LIBERO)
import subprocess
import time
import os
import torch
from pathlib import Path

LOGS_DIR = Path('/content/logs')
LOGS_DIR.mkdir(parents=True, exist_ok=True)

# Configuration
NUM_TRIALS = 5
LIBERO_BASE_PORT = 9001  # Ports: 9001-9005
METAWORLD_BASE_PORT = 9101  # Ports: 9101-9105

def cleanup_ports(ports):
    """Kill processes on specified ports"""
    for port in ports:
        subprocess.run(f"lsof -ti:{port} | xargs -r kill -9 2>/dev/null", shell=True)
    subprocess.run("pkill -f 'Evo1.*server' 2>/dev/null", shell=True)

def check_port(port, timeout=5):
    try:
        import websocket
        ws = websocket.create_connection(f"ws://localhost:{port}", timeout=timeout)
        ws.close()
        return True
    except Exception:
        return False

def wait_for_server(port, name, max_wait=180):
    print(f"  Waiting for {name} on port {port}...", end="", flush=True)
    start = time.time()
    while time.time() - start < max_wait:
        if check_port(port):
            print(f" ✅ ready ({int(time.time()-start)}s)")
            return True
        time.sleep(2)
        print(".", end="", flush=True)
    print(f" ❌ timeout")
    return False

def check_gpu_memory():
    """Return available GPU memory in GB"""
    result = subprocess.run(['nvidia-smi', '--query-gpu=memory.free', '--format=csv,noheader,nounits'],
                          capture_output=True, text=True)
    return int(result.stdout.strip()) / 1024

def monitor_processes(processes, benchmark_name):
    """Monitor processes and return when all complete"""
    try:
        while True:
            time.sleep(60)
            print('\n' + '='*60)
            print(f'--- {benchmark_name} Status ({time.strftime("%H:%M:%S")}) ---')
            print('='*60)

            all_done = True
            running_count = 0
            failed_count = 0
            done_count = 0

            # Group by type
            servers = [p for p in processes if p[0].startswith('server_')]
            clients = [p for p in processes if p[0].startswith('client_')]

            for group_name, group in [("Servers", servers), ("Clients", clients)]:
                print(f"\n{group_name}:")
                for name, p, _ in group:
                    if p.poll() is None:
                        print(f"  🟢 {name}: Running")
                        all_done = False
                        running_count += 1
                    else:
                        if p.returncode == 0:
                            print(f"  ✅ {name}: Done")
                            done_count += 1
                        else:
                            print(f"  ❌ {name}: Failed (code={p.returncode})")
                            failed_count += 1

            print(f"\n📊 Summary: {running_count} running, {done_count} done, {failed_count} failed")

            if all_done:
                print('\n' + '='*60)
                print(f'✅ {benchmark_name} completed!')
                print('='*60)
                break

    except KeyboardInterrupt:
        print(f'\n🛑 Stopping {benchmark_name}...')
        for name, p, log in processes:
            if p.poll() is None:
                print(f"  Terminating {name}...")
                p.terminate()
            log.close()
        raise

# ============================================================================
# PHASE 1: MetaWorld (5 parallel trials)
# ============================================================================

print('\n' + '='*80)
print('🎯 PHASE 1: MetaWorld Ablation Study (5 Parallel Trials)')
print('='*80)

# Cleanup
print('\n🧹 Cleaning up existing processes...')
metaworld_ports = list(range(METAWORLD_BASE_PORT, METAWORLD_BASE_PORT + NUM_TRIALS))
cleanup_ports(metaworld_ports)
time.sleep(2)

# Check GPU
free_mem = check_gpu_memory()
print(f'💾 GPU free memory: {free_mem:.1f}GB')

processes = []
env = {**os.environ, "PYTHONUNBUFFERED": "1"}

# Start MetaWorld Servers
print(f'\n📡 Starting {NUM_TRIALS} MetaWorld ablation servers...')
for i in range(1, NUM_TRIALS + 1):
    port = METAWORLD_BASE_PORT + i - 1
    trial_name = f"MetaWorld_Ablation_Trial_{i}"

    free_mem = check_gpu_memory()
    print(f"  [Trial {i}] GPU free: {free_mem:.1f}GB", end=" ")

    cmd = f"conda run --no-capture-output -n evo1_server python -u /content/Evo-1/Evo_1/scripts/Evo1_ablated_server.py --port {port} --checkpoint {CHECKPOINTS_DIR}/metaworld --name {trial_name}"

    log = open(f"{LOGS_DIR}/server_metaworld_ablation_trial_{i}.log", "w")
    p = subprocess.Popen(cmd, shell=True, stdout=log, stderr=log, env=env)
    processes.append((f"server_metaworld_{i}", p, log))

    if not wait_for_server(port, f"MW-{i}"):
        raise RuntimeError(f"MetaWorld server {i} failed to start on port {port}")

print(f'\n✅ All {NUM_TRIALS} MetaWorld servers ready!')

# Start MetaWorld Clients
print(f'\n🔬 Starting {NUM_TRIALS} MetaWorld clients...')
for i in range(1, NUM_TRIALS + 1 ):
    port = METAWORLD_BASE_PORT + i - 1
    patched_script = f'/content/Evo-1/MetaWorld_evaluation/mt50_evo1_client_trial_{i}.py'

    cmd = f"conda run --no-capture-output -n metaworld_client python -u {patched_script}"
    log = open(f"{LOGS_DIR}/client_metaworld_ablation_trial_{i}.log", "w")
    p = subprocess.Popen(cmd, shell=True, stdout=log, stderr=log, stdin=subprocess.DEVNULL, env=env)
    processes.append((f"client_metaworld_{i}", p, log))
    print(f'  ✅ MetaWorld client {i} started (port {port})')

print('\n' + '='*60)
print(f'✅ MetaWorld benchmarks running ({NUM_TRIALS} trials)')
print('='*60)
print(f'Logs: {LOGS_DIR}/server_metaworld_ablation_trial_*.log')
print(f'      {LOGS_DIR}/client_metaworld_ablation_trial_*.log')
print('\n⏳ Monitoring MetaWorld trials...')

# Monitor MetaWorld
monitor_processes(processes, "MetaWorld")

# Cleanup MetaWorld
print('\n🧹 Cleaning up MetaWorld servers...')
for name, p, log in processes:
    if p.poll() is None:
        p.terminate()
    log.close()

cleanup_ports(metaworld_ports)
time.sleep(3)

# Free GPU memory
print('🔄 Freeing GPU memory...')
torch.cuda.empty_cache()
time.sleep(2)

free_mem = check_gpu_memory()
print(f'💾 GPU free memory after cleanup: {free_mem:.1f}GB')



🎯 PHASE 1: MetaWorld Ablation Study (5 Parallel Trials)

🧹 Cleaning up existing processes...
💾 GPU free memory: 22.2GB

📡 Starting 5 MetaWorld ablation servers...
  [Trial 1] GPU free: 22.2GB   Waiting for MW-1 on port 9101.............. ✅ ready (22s)
  [Trial 2] GPU free: 20.1GB   Waiting for MW-2 on port 9102.......... ✅ ready (14s)
  [Trial 3] GPU free: 18.1GB   Waiting for MW-3 on port 9103.......... ✅ ready (14s)
  [Trial 4] GPU free: 16.1GB   Waiting for MW-4 on port 9104.......... ✅ ready (14s)
  [Trial 5] GPU free: 14.1GB   Waiting for MW-5 on port 9105.......... ✅ ready (14s)

✅ All 5 MetaWorld servers ready!

🔬 Starting 5 MetaWorld clients...
  ✅ MetaWorld client 1 started (port 9101)
  ✅ MetaWorld client 2 started (port 9102)
  ✅ MetaWorld client 3 started (port 9103)
  ✅ MetaWorld client 4 started (port 9104)
  ✅ MetaWorld client 5 started (port 9105)

✅ MetaWorld benchmarks running (5 trials)
Logs: /content/logs/server_metaworld_ablation_trial_*.log
      /content/logs/cl

KeyboardInterrupt: 

In [ ]:
from google.colab import drive
from pathlib import Path
import os

# Mount Drive
drive.mount('/content/drive')

In [ ]:
import os
import shutil
import zipfile

src_dir = "/content"
dst_dir = "/content/drive/MyDrive/Research/URECA/Results/MetaworldAblationPass0"
zip_path = "/content/metaworld_results.zip"

os.makedirs(dst_dir, exist_ok=True)

prefixes = ("metaworld_logs", "metaworld_video")

# Create zip file
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for name in os.listdir(src_dir):
        src_path = os.path.join(src_dir, name)

        if name.startswith(prefixes):
            if os.path.isdir(src_path):
                # Add directory to zip
                for root, dirs, files in os.walk(src_path):
                    for file in files:
                        file_path = os.path.join(root, file)
                        arcname = os.path.relpath(file_path, src_dir)
                        zipf.write(file_path, arcname)
            else:
                # Add file to zip
                zipf.write(src_path, name)

# Copy zip file to Drive
shutil.copy2(zip_path, os.path.join(dst_dir, "metaworld_results.zip"))

drive.flush_and_unmount()


In [ ]:
drive.flush_and_unmount()

In [ ]:
import subprocess
import time
import os
import torch
from pathlib import Path

LOGS_DIR = Path('/content/logs')
LOGS_DIR.mkdir(parents=True, exist_ok=True)

# Configuration
NUM_TRIALS = 10
LIBERO_BASE_PORT = 9001  # Ports: 9001-9005
METAWORLD_BASE_PORT = 9101  # Ports: 9101-9105

def cleanup_ports(ports):
    """Kill processes on specified ports"""
    for port in ports:
        subprocess.run(f"lsof -ti:{port} | xargs -r kill -9 2>/dev/null", shell=True)
    subprocess.run("pkill -f 'Evo1.*server' 2>/dev/null", shell=True)

def check_port(port, timeout=5):
    try:
        import websocket
        ws = websocket.create_connection(f"ws://localhost:{port}", timeout=timeout)
        ws.close()
        return True
    except Exception:
        return False

def wait_for_server(port, name, max_wait=180):
    print(f"  Waiting for {name} on port {port}...", end="", flush=True)
    start = time.time()
    while time.time() - start < max_wait:
        if check_port(port):
            print(f" ✅ ready ({int(time.time()-start)}s)")
            return True
        time.sleep(2)
        print(".", end="", flush=True)
    print(f" ❌ timeout")
    return False

def check_gpu_memory():
    """Return available GPU memory in GB"""
    result = subprocess.run(['nvidia-smi', '--query-gpu=memory.free', '--format=csv,noheader,nounits'],
                          capture_output=True, text=True)
    return int(result.stdout.strip()) / 1024

def monitor_processes(processes, benchmark_name):
    """Monitor processes and return when all complete"""
    try:
        while True:
            time.sleep(60)
            print('\n' + '='*60)
            print(f'--- {benchmark_name} Status ({time.strftime("%H:%M:%S")}) ---')
            print('='*60)

            all_done = True
            running_count = 0
            failed_count = 0
            done_count = 0

            # Group by type
            servers = [p for p in processes if p[0].startswith('server_')]
            clients = [p for p in processes if p[0].startswith('client_')]

            for group_name, group in [("Servers", servers), ("Clients", clients)]:
                print(f"\n{group_name}:")
                for name, p, _ in group:
                    if p.poll() is None:
                        print(f"  🟢 {name}: Running")
                        all_done = False
                        running_count += 1
                    else:
                        if p.returncode == 0:
                            print(f"  ✅ {name}: Done")
                            done_count += 1
                        else:
                            print(f"  ❌ {name}: Failed (code={p.returncode})")
                            failed_count += 1

            print(f"\n📊 Summary: {running_count} running, {done_count} done, {failed_count} failed")

            if all_done:
                print('\n' + '='*60)
                print(f'✅ {benchmark_name} completed!')
                print('='*60)
                break

    except KeyboardInterrupt:
        print(f'\n🛑 Stopping {benchmark_name}...')
        for name, p, log in processes:
            if p.poll() is None:
                print(f"  Terminating {name}...")
                p.terminate()
            log.close()
        raise

# ============================================================================
# PHASE 2: LIBERO (5 parallel trials)
# ============================================================================

print('\n' + '='*80)
print('🎯 PHASE 2: LIBERO Ablation Study (5 Parallel Trials)')
print('='*80)

# Cleanup
print('\n🧹 Cleaning up existing processes...')
libero_ports = list(range(LIBERO_BASE_PORT, LIBERO_BASE_PORT + NUM_TRIALS))
cleanup_ports(libero_ports)
time.sleep(2)

# Check GPU
free_mem = check_gpu_memory()
print(f'💾 GPU free memory: {free_mem:.1f}GB')

processes = []

# Start LIBERO Servers
print(f'\n📡 Starting {NUM_TRIALS} LIBERO ablation servers...')
for i in range(1, NUM_TRIALS + 1):
    port = LIBERO_BASE_PORT + i - 1
    trial_name = f"LIBERO_Ablation_Trial_{i}"

    free_mem = check_gpu_memory()
    print(f"  [Trial {i}] GPU free: {free_mem:.1f}GB", end=" ")

    cmd = f"conda run --no-capture-output -n evo1_server python -u /content/Evo-1/Evo_1/scripts/Evo1_ablated_server.py --port {port} --checkpoint {CHECKPOINTS_DIR}/libero --name {trial_name}"

    log = open(f"{LOGS_DIR}/server_libero_ablation_trial_{i}.log", "w")
    p = subprocess.Popen(cmd, shell=True, stdout=log, stderr=log, env=env)
    processes.append((f"server_libero_{i}", p, log))

    if not wait_for_server(port, f"LIBERO-{i}"):
        raise RuntimeError(f"LIBERO server {i} failed to start on port {port}")

print(f'\n✅ All {NUM_TRIALS} LIBERO servers ready!')

# Start LIBERO Clients
print(f'\n🔬 Starting {NUM_TRIALS} LIBERO clients...')
for i in range(1, NUM_TRIALS + 1):
    port = LIBERO_BASE_PORT + i - 1
    patched_script = f'/content/Evo-1/LIBERO_evaluation/libero_client_trial_{i}.py'

    cmd = f"conda run --no-capture-output -n libero_client python -u {patched_script}"
    log = open(f"{LOGS_DIR}/client_libero_ablation_trial_{i}.log", "w")
    client_env = {**env, "PYTHONPATH": "/content/LIBERO_evaluation/LIBERO"}
    p = subprocess.Popen(cmd, shell=True, stdout=log, stderr=log, stdin=subprocess.DEVNULL, env=client_env)
    processes.append((f"client_libero_{i}", p, log))
    print(f'  ✅ LIBERO client {i} started (port {port})')

print('\n' + '='*60)
print(f'✅ LIBERO benchmarks running ({NUM_TRIALS} trials)')
print('='*60)
print(f'Logs: {LOGS_DIR}/server_libero_ablation_trial_*.log')
print(f'      {LOGS_DIR}/client_libero_ablation_trial_*.log')
print('\n⏳ Monitoring LIBERO trials...')

# Monitor LIBERO
monitor_processes(processes, "LIBERO")

# Final cleanup
print('\n🧹 Final cleanup...')
for name, p, log in processes:
    if p.poll() is None:
        p.terminate()
    log.close()

cleanup_ports(libero_ports)

# ============================================================================
# Summary
# ============================================================================

print('\n' + '='*80)
print('📊 ABLATION STUDY COMPLETE')
print('='*80)
print(f'\n✅ Phase 1: MetaWorld ({NUM_TRIALS} trials) - COMPLETE')
print(f'✅ Phase 2: LIBERO ({NUM_TRIALS} trials) - COMPLETE')
print(f'\n📁 All logs saved to: {LOGS_DIR}/')
print(f'\nMetaWorld:')
print(f'  - Server logs: server_metaworld_ablation_trial_{{1-{NUM_TRIALS}}}.log')
print(f'  - Client logs: client_metaworld_ablation_trial_{{1-{NUM_TRIALS}}}.log')
print(f'  - Videos: /content/metaworld_videos_trial_{{1-{NUM_TRIALS}}}/')
print(f'\nLIBERO:')
print(f'  - Server logs: server_libero_ablation_trial_{{1-{NUM_TRIALS}}}.log')
print(f'  - Client logs: client_libero_ablation_trial_{{1-{NUM_TRIALS}}}.log')
print('\n💡 Next steps:')
print('  1. Review logs for any errors')
print('  2. Compare with baseline results')
print('  3. Analyze performance degradation due to state ablation')
print('='*80)

Streaming output truncated to the last 5000 lines.
  🟢 client_libero_5: Running
  🟢 client_libero_6: Running
  🟢 client_libero_7: Running
  🟢 client_libero_8: Running
  🟢 client_libero_9: Running
  🟢 client_libero_10: Running

📊 Summary: 20 running, 0 done, 0 failed

--- LIBERO Status (16:14:16) ---

Servers:
  🟢 server_libero_1: Running
  🟢 server_libero_2: Running
  🟢 server_libero_3: Running
  🟢 server_libero_4: Running
  🟢 server_libero_5: Running
  🟢 server_libero_6: Running
  🟢 server_libero_7: Running
  🟢 server_libero_8: Running
  🟢 server_libero_9: Running
  🟢 server_libero_10: Running

Clients:
  🟢 client_libero_1: Running
  🟢 client_libero_2: Running
  🟢 client_libero_3: Running
  🟢 client_libero_4: Running
  🟢 client_libero_5: Running
  🟢 client_libero_6: Running
  🟢 client_libero_7: Running
  🟢 client_libero_8: Running
  🟢 client_libero_9: Running
  🟢 client_libero_10: Running

📊 Summary: 20 running, 0 done, 0 failed

--- LIBERO Status (16:15:16) ---

Servers:
  🟢 server_l

KeyboardInterrupt: 

In [ ]:
import os
import shutil
import zipfile

src_dir = "/content"
dst_dir = "/content/drive/MyDrive/Research/URECA/Results/LIBEROAblationPass0"
zip_path = "/content/LIBERO_ablation.zip"

os.makedirs(dst_dir, exist_ok=True)

prefixes = ("logs", "video", "log_file")

# Create zip file
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for name in os.listdir(src_dir):
        src_path = os.path.join(src_dir, name)

        if name.startswith(prefixes):
            if os.path.isdir(src_path):
                # Add directory to zip
                for root, dirs, files in os.walk(src_path):
                    for file in files:
                        file_path = os.path.join(root, file)
                        arcname = os.path.relpath(file_path, src_dir)
                        zipf.write(file_path, arcname)
            else:
                # Add file to zip
                zipf.write(src_path, name)

# Copy zip file to Drive
shutil.copy2(zip_path, os.path.join(dst_dir, "metaworld_results.zip"))

drive.flush_and_unmount()


In [ ]:
# 8. View results
print('📊 Ablation Study Results\n')

print('--- LIBERO (State Encoder Ablated) ---')
!tail -30 {LOGS_DIR}/client_libero.log 2>/dev/null || echo 'No results yet'

print('\n--- MetaWorld (State Encoder Ablated) ---')
!tail -30 {LOGS_DIR}/client_metaworld.log 2>/dev/null || echo 'No results yet'

In [ ]:
# 9. Save results to Drive
import shutil
from datetime import datetime
import pathlib as Path

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
results_subdir = RESULTS_DIR / f'ablation_state_encoder_{timestamp}'
results_subdir.mkdir(parents=True, exist_ok=True)

print(f'💾 Saving results to Drive...')

# Copy all logs
for log_file in LOGS_DIR.glob('*.log'):
    shutil.copy(log_file, results_subdir / log_file.name)
    print(f'  ✅ {log_file.name}')

# Create summary
summary = f'''# State Encoder Ablation Results
Timestamp: {timestamp}

## Configuration
- Ablation: Zero-state input (vision-only policy)
  - State input: All zeros passed to model
  - Tests: Can vision alone solve manipulation tasks without proprioceptive state?
- Checkpoints: LIBERO ({CHECKPOINTS_DIR}/libero), MetaWorld ({CHECKPOINTS_DIR}/metaworld)
- Environment: Conda (evo1_ablation)

## Ablation Implementation
Simple zero-state ablation: Normalizer returns zeros instead of normalized state values.
Model architecture unchanged - integration module still runs, just receives zero state input.

## Results
See log files for detailed metrics.

## Comparison
Compare against baseline (colab_benchmark_clean.ipynb) to quantify state encoder importance.
'''

with open(results_subdir / 'README.md', 'w') as f:
    f.write(summary)

print(f'\n✅ Results saved to: {results_subdir}')
print('\nTo download:')
print(f'  from google.colab import files')
print(f'  !zip -r ablation_results.zip {results_subdir}')
print(f'  files.download("ablation_results.zip")')